# Day 37: Pydantic-based Schema for Tool Calling & Integration

Welcome to Day 37! In this module, we will explore how to explicitly define tools for our Large Language Models using **Pydantic**. We will define a strict schema for a search tool and integrate it seamlessly.

## Core Theory (Just-in-Time)

### Why Pydantic for Tool Calling?
Tool calling (or Function calling) allows LLMs to interact with external systems. To make this reliable, the LLM needs a strict schema defining what tools are available, what arguments they take, and their types.
Pydantic provides:
- **Strict Type Validation:** Ensures the arguments LLM attempts to pass actually match the expected types (e.g., passing a string when an int is required will fail gracefully).
- **Automatic Schema Generation:** Pydantic models can automatically be exported as JSON Schemas which is exactly what APIs like OpenAI, Anthropic, or Groq expect for their tool definitions.
- **Self-Documenting Code:** Clean class structures with docstrings to provide semantic meaning to the LLM.

### The Flow
1. **Define Schema:** Create a Pydantic `BaseModel` detailing the tool's input structure.
2. **Implement Tool Logic:** Write the Python function that performs the actual operation.
3. **Bind Tool to LLM:** Use a framework (like LangChain's `@tool` decorator or `bind_tools`) to link the function and its schema to the model.
4. **Execution & Parsing:** The LLM decides to use the tool, outputs arguments matching the schema, and the framework executes the underlying logic.

In [1]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.tools import tool
import json

class SearchInputSchema(BaseModel):
    """Schema defining the inputs for the search tool."""
    query: str = Field(
        ..., 
        description="The specific search query or keyword to look up."
    )
    num_results: int = Field(
        default=3, 
        description="The number of results to return. Default is 3."
    )

@tool(args_schema=SearchInputSchema)
def web_search_tool(query: str, num_results: int = 3) -> str:
    """Search the web for information based on a query."""
    # In production, this would call an API like Tavily, Google Custom Search, etc.
    # Here, we mock the tool logic.
    print(f"Executing search for: '{query}', max results: {num_results}")
    mock_results = [
        {"title": f"Result 1 for {query}", "content": "This is a mock description..."},
        {"title": f"Result 2 for {query}", "content": "Another relevant document..."}
    ]
    return json.dumps(mock_results[:num_results])

print("Tool Name:", web_search_tool.name)
print("Tool Description:", web_search_tool.description)
print("Tool Schema:", json.dumps(web_search_tool.args_schema.model_json_schema(), indent=2))


Tool Name: web_search_tool
Tool Description: Search the web for information based on a query.
Tool Schema: {
  "description": "Schema defining the inputs for the search tool.",
  "properties": {
    "query": {
      "description": "The specific search query or keyword to look up.",
      "title": "Query",
      "type": "string"
    },
    "num_results": {
      "default": 3,
      "description": "The number of results to return. Default is 3.",
      "title": "Num Results",
      "type": "integer"
    }
  },
  "required": [
    "query"
  ],
  "title": "SearchInputSchema",
  "type": "object"
}


## Integrating the Tool with an LLM

Now let's bind this tool to a LangChain model. We'll use the `init_chat_model` utility or direct instantiation, binding the tool, and letting the model decide to invoke it.

In [2]:
import os
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

# We can manually inspect what the tool conversion looks like:
openai_tool_format = convert_to_openai_tool(web_search_tool)
print("Converted OpenAI Tool Format:")
print(json.dumps(openai_tool_format, indent=2))

# Note: To actually run a model locally you need an API Key.
# The below code snippet shows how you would bind and invoke it.
try:
    # This requires langchain-openai and OPENAI_API_KEY set
    from langchain_openai import ChatOpenAI
    
    if not os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENAI_API_KEY") == "sk-dummy-key":
        print("\n[MOCK] Skipping actual LLM invocation due to missing/dummy API key.")
    else:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        # Bind the tool to the LLM
        llm_with_tools = llm.bind_tools([web_search_tool])
        
        response = llm_with_tools.invoke([HumanMessage(content="Find the latest news on AI.")])
        print("\nLLM Response (Tool Calls):", response.tool_calls)
except ImportError:
    print("\n[MOCK] Please install langchain-openai to test actual API invocation.")


Converted OpenAI Tool Format:
{
  "type": "function",
  "function": {
    "name": "web_search_tool",
    "description": "Search the web for information based on a query.",
    "parameters": {
      "properties": {
        "query": {
          "description": "The specific search query or keyword to look up.",
          "type": "string"
        },
        "num_results": {
          "default": 3,
          "description": "The number of results to return. Default is 3.",
          "type": "integer"
        }
      },
      "required": [
        "query"
      ],
      "type": "object"
    }
  }
}

[MOCK] Please install langchain-openai to test actual API invocation.


## Common Pitfalls in Production

1.  **Vague Field Descriptions:** LLMs rely heavily on the `description` string inside `Field()`. If it's ambiguous, the LLM will hallucinate arguments or fail to call the tool correctly.
2.  **Missing Defaults:** If an argument is optional, explicitly provide a `default=` value in the Pydantic schema so the LLM isn't forced to guess a value when it's irrelevant.
3.  **Complex Nested Schemas:** Highly nested or recursive schemas confuse smaller models. Keep the input schema flat and straightforward.
4.  **Error Handling (No Fallback):** When the tool logic throws an exception (e.g., API timeout), the LLM chain might break. Always wrap tool execution in a `try/except` and return a stringified error message back to the LLM so it can attempt to correct its behavior.

## Practical Lab / Homework

**Your Task:**
Create a new tool named `fetch_weather_tool`.
1. Define a Pydantic schema named `WeatherInputSchema`.
    - It should require a `location` (string, description: "The city and state, e.g., 'San Francisco, CA'").
    - It should optionally take a `unit` (string, restricted to 'celsius' or 'fahrenheit', default 'fahrenheit').
2. Implement the `@tool` logic returning a mocked JSON string of weather data.
3. Print the generated JSON schema to verify your schema configuration.

In [3]:
from typing import Literal

# 1. Define the Pydantic Schema
class WeatherInputSchema(BaseModel):
    """Schema for fetching weather data."""
    location: str = Field(
        ...,
        description="The city and state, e.g., 'San Francisco, CA'"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="fahrenheit",
        description="The temperature unit to use. Either 'celsius' or 'fahrenheit'."
    )

# 2. Implement the Tool
@tool(args_schema=WeatherInputSchema)
def fetch_weather_tool(location: str, unit: str = "fahrenheit") -> str:
    """Fetch the current weather for a specified location."""
    print(f"Fetching weather for {location} in {unit}")
    mock_data = {
        "location": location,
        "temperature": 72 if unit == "fahrenheit" else 22,
        "condition": "Sunny",
        "unit": unit
    }
    return json.dumps(mock_data)

# 3. Verify Schema
print("Weather Tool Schema:\n", json.dumps(fetch_weather_tool.args_schema.model_json_schema(), indent=2))


Weather Tool Schema:
 {
  "description": "Schema for fetching weather data.",
  "properties": {
    "location": {
      "description": "The city and state, e.g., 'San Francisco, CA'",
      "title": "Location",
      "type": "string"
    },
    "unit": {
      "default": "fahrenheit",
      "description": "The temperature unit to use. Either 'celsius' or 'fahrenheit'.",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "title": "Unit",
      "type": "string"
    }
  },
  "required": [
    "location"
  ],
  "title": "WeatherInputSchema",
  "type": "object"
}
